In [1]:
#Uma vez descoberto o padrão de organização dos dados de Matão
#Partimos para a seperação de todos os dados relativos a Matão

import pandas as pd
import numpy as np
import os

# Lista com os caminhos dos 3 arquivos de sinistros
arquivos_sinistros = [
    '../data/sinistros_2015-2021.csv',
    '../data/sinistros_2022-2024.csv',
    '../data/sinistros_2025-2026.csv'
]

codigo_ibge_matao = 3529302
lista_df_matao = []

print("Iniciando carregamento e filtragem de Matão...")

for arquivo in arquivos_sinistros:
    nome_arquivo = os.path.basename(arquivo)
    registros_arquivo = 0
    
    # Leitura em lotes de 20.000 linhas
    for chunk in pd.read_csv(arquivo, sep=';', encoding='latin1', chunksize=20000, low_memory=False):
        chunk_matao = chunk[chunk['cod_ibge'] == codigo_ibge_matao]
        if not chunk_matao.empty:
            lista_df_matao.append(chunk_matao)
            registros_arquivo += len(chunk_matao)
            
    print(f" -> {nome_arquivo}: {registros_arquivo} registros encontrados.")

# Consolidação em um único DataFrame
df_matao_bruto = pd.concat(lista_df_matao, ignore_index=True)

print("\n--- Consolidação Concluída ---")
print(f"Total geral de registros (2015-2026): {len(df_matao_bruto)}")
print(f"Total de colunas: {df_matao_bruto.shape[1]}")
print(f"Uso de memória: {df_matao_bruto.memory_usage().sum() / 1024:.2f} KB\n")

print("--- Distribuição de Sinistros por Ano ---")
print(df_matao_bruto['ano_sinistro'].value_counts().sort_index())


Iniciando carregamento e filtragem de Matão...
 -> sinistros_2015-2021.csv: 1408 registros encontrados.
 -> sinistros_2022-2024.csv: 1748 registros encontrados.
 -> sinistros_2025-2026.csv: 918 registros encontrados.

--- Consolidação Concluída ---
Total geral de registros (2015-2026): 4074
Total de colunas: 50
Uso de memória: 1591.54 KB

--- Distribuição de Sinistros por Ano ---
ano_sinistro
2015     12
2016     14
2017      6
2018     10
2019    470
2020    410
2021    486
2022    497
2023    678
2024    573
2025    547
2026    371
Name: count, dtype: int64


In [2]:
# Passo 4.2: Seleção e Renomeação das Colunas Principais

mapa_colunas = {
    # Identificação e Temporal
    'id_sinistro': 'id_sinistro',
    'tipo_registro': 'tipo_registro',
    'data_sinistro': 'data',
    'ano_sinistro': 'ano',
    'mes_sinistro': 'mes',
    'dia_sinistro': 'dia',
    'hora_sinistro': 'hora',
    'dia_da_semana': 'dia_semana',
    'turno': 'turno',
    
    # Localização
    'tipo_via': 'tipo_via',
    'tipo_local': 'tipo_local',
    'logradouro': 'logradouro',
    'latitude': 'latitude',
    'longitude': 'longitude',
    
    # Tipo de Sinistro
    'tp_sinistro_primario': 'tipo_sinistro',
    
    # Veículos Envolvidos
    'qtd_bicicleta': 'qtd_bicicleta',
    'qtd_motocicleta': 'qtd_motocicleta',
    'qtd_automovel': 'qtd_automovel',
    'qtd_onibus': 'qtd_onibus',
    'qtd_caminhao': 'qtd_caminhao',
    'qtd_pedestre': 'qtd_pedestre',
    'qtd_veic_outros': 'qtd_outros_veiculos',
    
    # Gravidade / Vítimas
    'qtd_gravidade_fatal': 'mortos',
    'qtd_gravidade_grave': 'feridos_graves',
    'qtd_gravidade_leve': 'feridos_leves',
    'qtd_gravidade_ileso': 'ilesos'
}

# Criando cópia apenas com as colunas mapeadas e renomeadas
df_matao = df_matao_bruto[list(mapa_colunas.keys())].rename(columns=mapa_colunas).copy()

print(f"Novo formato: {df_matao.shape[0]} linhas e {df_matao.shape[1]} colunas selecionadas.")
print("\nPrimeiras 3 linhas do dataset selecionado:")
df_matao.head(3)


Novo formato: 4074 linhas e 26 colunas selecionadas.

Primeiras 3 linhas do dataset selecionado:


,id_sinistro,tipo_registro,data,ano,mes,dia,hora,dia_semana,turno,tipo_via,...,qtd_motocicleta,qtd_automovel,qtd_onibus,qtd_caminhao,qtd_pedestre,qtd_outros_veiculos,mortos,feridos_graves,feridos_leves,ilesos
0,2456417,SINISTRO FATAL,19/03/2015,2015,3,19,21:53,Quinta-feira,NOITE,VIAS URBANAS,...,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN
1,2456409,SINISTRO FATAL,18/06/2015,2015,6,18,20:15,Quinta-feira,NOITE,ESTRADAS E RODOVIAS,...,NaN,2.0,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN
2,2456288,SINISTRO FATAL,03/10/2015,2015,10,3,03:10,Sábado,MADRUGADA,VIAS URBANAS,...,NaN,2.0,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN


In [3]:
# Passos 4.3 e 4.4: Tratamento de Nulos e Conversão de Tipos

# 1. Tratar contagens de veículos e vítimas (NaN -> 0 -> int)
colunas_contagens = [
    'qtd_bicicleta', 'qtd_motocicleta', 'qtd_automovel', 
    'qtd_onibus', 'qtd_caminhao', 'qtd_pedestre', 'qtd_outros_veiculos',
    'mortos', 'feridos_graves', 'feridos_leves', 'ilesos'
]

for col in colunas_contagens:
    df_matao[col] = df_matao[col].fillna(0).astype(int)

# 2. Converter coluna de data para datetime
df_matao['data'] = pd.to_datetime(df_matao['data'], format='%d/%m/%Y', errors='coerce')

# 3. Extrair a hora inteira (0 a 23) a partir do horário
df_matao['hora_int'] = pd.to_datetime(df_matao['hora'], format='%H:%M', errors='coerce').dt.hour

# 4. Tratar Latitude e Longitude (trocar vírgula por ponto e converter para float)
df_matao['latitude'] = pd.to_numeric(df_matao['latitude'].astype(str).str.replace(',', '.'), errors='coerce')
df_matao['longitude'] = pd.to_numeric(df_matao['longitude'].astype(str).str.replace(',', '.'), errors='coerce')

# --- Verificação da Saúde dos Dados ---
print("--- Verificação dos Tipos de Dados Convertidos ---")
print(df_matao[['data', 'hora_int', 'latitude', 'longitude', 'mortos', 'qtd_motocicleta']].dtypes)

print("\n--- Validações de Conversão ---")
print(f"Período das datas: de {df_matao['data'].min().strftime('%d/%m/%Y')} até {df_matao['data'].max().strftime('%d/%m/%Y')}")
print(f"Total de registros com coordenadas válidas: {df_matao['latitude'].notnull().sum()} de {len(df_matao)}")
print(f"Horas convertidas com sucesso: {df_matao['hora_int'].notnull().sum()} de {len(df_matao)}")


--- Verificação dos Tipos de Dados Convertidos ---
data               datetime64[us]
hora_int                  float64
latitude                  float64
longitude                 float64
mortos                      int64
qtd_motocicleta             int64
dtype: object

--- Validações de Conversão ---
Período das datas: de 18/03/2015 até 31/08/2026
Total de registros com coordenadas válidas: 3654 de 4074
Horas convertidas com sucesso: 4071 de 4074


In [4]:
# Passo 4.5: Criação de Colunas Derivadas (Feature Engineering)

# 1. Total de vítimas por ocorrência
df_matao['total_vitimas'] = (
    df_matao['feridos_leves'] + df_matao['feridos_graves'] + df_matao['mortos']
)

# 2. Total de veículos envolvidos
df_matao['total_veiculos'] = (
    df_matao['qtd_bicicleta'] + df_matao['qtd_motocicleta'] + 
    df_matao['qtd_automovel'] + df_matao['qtd_onibus'] + 
    df_matao['qtd_caminhao'] + df_matao['qtd_outros_veiculos']
)

# 3. Classificação simplificada e vetorizada de Gravidade
condicoes = [
    df_matao['mortos'] > 0,
    df_matao['feridos_graves'] > 0,
    df_matao['feridos_leves'] > 0
]
rotulos = [
    'Com óbito',
    'Com feridos graves',
    'Com feridos leves'
]
df_matao['gravidade'] = np.select(condicoes, rotulos, default='Sem vítimas')

# 4. Indicador de fim de semana (Sábado ou Domingo)
df_matao['fim_de_semana'] = df_matao['dia_semana'].astype(str).str.upper().str.contains('SÁBADO|SABADO|DOMINGO')

# --- Verificando os resultados criados ---
print("--- Distribuição da Gravidade dos Sinistros em Matão ---")
print(df_matao['gravidade'].value_counts())
print("\nPercentual:")
print((df_matao['gravidade'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')

print("\n--- Estatísticas do Total de Vítimas ---")
print(df_matao[['total_vitimas', 'total_veiculos']].describe())


--- Distribuição da Gravidade dos Sinistros em Matão ---
gravidade
Sem vítimas           1920
Com feridos leves     1726
Com feridos graves     286
Com óbito              142
Name: count, dtype: int64

Percentual:
gravidade
Sem vítimas           47.13%
Com feridos leves     42.37%
Com feridos graves     7.02%
Com óbito              3.49%
Name: proportion, dtype: str

--- Estatísticas do Total de Vítimas ---
       total_vitimas  total_veiculos
count    4074.000000     4074.000000
mean        0.647030        0.912126
std         0.728641        0.986098
min         0.000000        0.000000
25%         0.000000        0.000000
50%         1.000000        1.000000
75%         1.000000        2.000000
max         7.000000       11.000000


In [5]:
# Passo 4.6: Exportação e Validação do Arquivo Final

caminho_saida = '../data/matao_limpo.csv'

# Salvar sem o índice numérico do pandas
df_matao.to_csv(caminho_saida, index=False)

# Teste de leitura imediata para garantir integridade do arquivo gerado
df_teste = pd.read_csv(caminho_saida)

print("--- Sucesso na Exportação ---")
print(f"Arquivo salvo com sucesso em: {caminho_saida}")
print(f"Total de linhas salvas: {len(df_teste)}")
print(f"Total de colunas salvas: {df_teste.shape[1]}")
print(f"Tamanho do arquivo em disco: {os.path.getsize(caminho_saida) / 1024:.2f} KB")


--- Sucesso na Exportação ---
Arquivo salvo com sucesso em: ../data/matao_limpo.csv
Total de linhas salvas: 4074
Total de colunas salvas: 31
Tamanho do arquivo em disco: 788.63 KB
